# Semantic layer

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/semantic-layer.ipynb)

A semantic layer defines reusable business metrics, dimensions and join rules. It cannot repair an incorrect underlying grain. Start with an explicit numerator, denominator, time basis and cancellation policy.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {'duckdb': 'duckdb>=1.0', 'dbt': 'dbt-core>=1.9,<2', 'dbt.adapters.duckdb': 'dbt-duckdb>=1.9,<2'}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Decisions and trade-offs

Define revenue before defining a dashboard: gross or net, paid or booked, order date or refund date, transaction currency or reporting currency. Ratios usually need weighted aggregation: total revenue divided by total orders, not an unweighted average of regional averages.

Version metric definitions and test them against small fixtures shared with stakeholders. The example below uses real DuckDB SQL; it does not run a vendor semantic service.

[dbt Semantic Layer](https://docs.getdbt.com/docs/use-dbt-semantic-layer/dbt-sl)

## Worked example

Run the small fixture and inspect each assertion. Change one input to see which contract fails.

In [ ]:
import duckdb
with duckdb.connect() as con:
    rows=con.sql("select * from (values ('A',100,1),('B',90,9)) t(region,revenue,orders)")
    rows.create_view('metrics')
    assert con.sql('select sum(revenue)*1.0/sum(orders) from metrics').fetchall()[0]==(19.0,)
    assert con.sql('select avg(revenue*1.0/orders) from metrics').fetchall()[0]==(55.0,)

## Practice

Add a failure case, explain the expected outcome before running it, and identify which guarantee would need a real service or a larger dataset to verify.